In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from statsmodels.tsa.stattools import ccf
import plotly.express as px

In [ ]:
# 导入csv数据，文件路径是上一级文件夹里的data文件夹下的data_AutoReg.csv
df = pd.read_csv('./data/data_AutoReg.csv')

# 将date这一列转换成pandas的时间格式，方便后续按时间处理、插值
df['date'] = pd.to_datetime(df['date'])
# 把date设置为索引，并且按照时间顺序排序（防止原始数据时间乱序）
df = df.set_index('date').sort_index()

# 打印数据基本信息：列名、类型、非空数量、内存占用
print(df.info())
# 打印统计描述，.T代表转置，方便查看每一列的均值、标准差、最大最小值等
print(df.describe().T)

# 把列名里面的空格全部删掉
df.columns = df.columns.str.replace(' ', '')  
# 把列名里面的#符号全部删掉
df.columns = df.columns.str.replace('#', '')  

# 时间插值：根据索引（时间）的间隔做线性插值补缺失值，时间间隔不一样也能正确加权
df_interp = df.interpolate(method='time')

# 筛选出所有以PT_8313A开头的列（A组炉膛压力测点）
cols_pressure = [col for col in df.columns if col.startswith("PT_8313A")]
# 画出A组所有压力测点时序曲线，画布大小15*5，设置图标题
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors A")
# 展示图片（matplotlib）
plt.show()

# 筛选B组压力测点
cols_pressure = [col for col in df.columns if col.startswith("PT_8313B")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors B")
plt.show()

# 筛选C组压力测点
cols_pressure = [col for col in df.columns if col.startswith("PT_8313C")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors C")
plt.show()

# 筛选D组压力测点
cols_pressure = [col for col in df.columns if col.startswith("PT_8313D")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors D")
plt.show()

# 筛选E组压力测点
cols_pressure = [col for col in df.columns if col.startswith("PT_8313E")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors E")
plt.show()

# 筛选F组压力测点
cols_pressure = [col for col in df.columns if col.startswith("PT_8313F")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors F")
plt.show()

# 筛选全部PT_8313开头的测点（A~F全部炉膛压力放一张图对比）
cols_pressure = [col for col in df.columns if col.startswith("PT_8313")]
df_interp[cols_pressure].plot(figsize=(15,5), title="Upper Furnace Pressure Sensors")
plt.show()


In [ ]:
# 导入plotly绘图模块go，用来绘制交互式图表
import plotly.graph_objects as go

# 创建一个空白画布（图对象），后续往里面添加曲线
fig = go.Figure()

# 向画布添加第一条曲线
fig.add_trace(go.Scatter(
    # x轴：时间索引
    x=df_interp.index,
    # y轴：一次空预器出口温度测点 TE_8303.AV_0
    y=df_interp["TE_8303.AV_0"],
    # 绘图模式：只画连线，不画散点标记
    mode='lines',
    # 图例上显示这条曲线的名字
    name='Primary air preheater outlet air temperature'
))

# 向画布添加第二条曲线
fig.add_trace(go.Scatter(
    # x轴同样使用时间索引
    x=df_interp.index,
    # y轴：炉膛上部压力A测点 PT_8313A.AV_0
    y=df_interp["PT_8313A.AV_0"],
    # 只画连线
    mode='lines',
    # 图例名称
    name='Upper furnace pressure (A)'
))

# 更新整张图的布局配置（标题、坐标轴、图例、画布大小等）
fig.update_layout(
    # 图表主标题
    title="📊 Temperature and Pressure Time Series",
    # x轴名称
    xaxis_title="Time",
    # y轴名称
    yaxis_title="Sensor Value",
    # 图例位置：x=0.01靠左，y=0.99靠上
    legend=dict(x=0.01, y=0.99),
    # 图表整体高度，单位像素
    height=500
)

# 在浏览器/输出窗口展示交互式图片，可以缩放、悬停看数值
fig.show()


In [ ]:
import plotly.graph_objects as go

# 创建画布
fig = go.Figure()

# 第一条曲线：温度，绑定左侧Y轴（默认yaxis）
fig.add_trace(go.Scatter(
    x=df_interp.index,
    y=df_interp["TE_8303.AV_0"],
    mode='lines',
    name='Primary air preheater outlet air temperature'
))

# 第二条曲线：压力，绑定右侧Y轴（yaxis2）
fig.add_trace(go.Scatter(
    x=df_interp.index,
    y=df_interp["PT_8313A.AV_0"],
    mode='lines',
    name='Upper furnace pressure (A)',
    yaxis="y2"   # 指定使用第二个y轴
))

# 布局设置，开启双Y轴
fig.update_layout(
    title="📊 Temperature and Pressure Time Series",
    xaxis_title="Time",
    # 左侧Y轴（温度）
    yaxis=dict(title="Temperature"),
    # 右侧Y轴（压力）
    yaxis2=dict(
        title="Pressure",
        overlaying="y",  # 和y轴共享x轴
        side="right"     # 放在右边
    ),
    legend=dict(x=0.01, y=0.99),
    height=500
)

fig.show()


In [ ]:
# 定义Z-score异常检测函数
# series：输入一维时序数据；threshold：Z分数阈值，默认3（3σ准则）
def detect_zscore_anomalies(series, threshold=3):
    # 先删掉序列里的空值，再计算每个点的z分数
    z_scores = zscore(series.dropna())
    # 绝对值大于阈值的标记为True（异常点），返回布尔数组
    return np.abs(z_scores) > threshold

# 循环遍历每一路压力测点
for col in cols_pressure:
    # 调用异常检测函数，得到该列的异常标记布尔序列
    anomalies = detect_zscore_anomalies(df_interp[col])
    # 创建画布，大小宽12，高4
    plt.figure(figsize=(12, 4))
    # 绘制时序曲线：x是时间索引，y是传感器数值，图例名称为测点名
    plt.plot(df_interp.index, df_interp[col], label=col)
    # 红色散点画出异常点，s=10是散点大小
    plt.scatter(df_interp.index[anomalies], df_interp[col][anomalies], color='red', label='Anomaly', s=10)
    # 显示图例
    plt.legend()
    # 设置图标题，显示当前测点名称
    plt.title(f"Anomalies in {col}")
    # 展示图片
    plt.show()


In [ ]:
# 导入statsmodels里的互相关函数ccf（这段代码下面并没有用到ccf）
from statsmodels.tsa.stattools import ccf

# 取压力信号PT_8313A.AV_0，前向填充空值（用上一个有效值补齐NaN）
x = df_interp["PT_8313A.AV_0"].ffill()
# 取温度信号TE_8332A.AV_0，同样前向填充空值
y = df_interp["TE_8332A.AV_0"].ffill()

# 生成滞后序列：从-100到+100，一共201个滞后阶数
# lag负数：x滞后y；lag正数：y滞后x
lags = np.arange(-100, 101)
# 循环每个lag：把x平移 -lag 之后，和y计算相关系数，得到互相关序列
corrs = [x.shift(-lag).corr(y) for lag in lags]

# 创建画布，宽12，高4
plt.figure(figsize=(12,4))
# 绘制滞后-相关系数曲线
plt.plot(lags, corrs)
# 在lag=0位置画红色虚线（代表无滞后对齐）
plt.axvline(0, color='red', linestyle='--')
# 设置图标题：PT_8313A压力和TE_8332A温度的互相关图
plt.title("Cross-correlation between PT_8313A and TE_8332A")
# x轴标签：滞后步数（采样点个数）
plt.xlabel("Lag (samples)")
# y轴标签：相关系数
plt.ylabel("Correlation")
# 打开网格线，方便读数
plt.grid(True)
# 展示图像
plt.show()


In [ ]:
# 导入sklearn的孤立森林算法，用于多维异常检测
from sklearn.ensemble import IsolationForest

# 挑选参与建模的传感器测点（多维特征）
sensor_cols = [
    "TE_8303.AV_0", "TE_8332A.AV_0", "PT_8313A.AV_0",
    "FT_8301.AV_0", "FT_8302.AV_0"
]

# 取出选定测点的数据，并删除还有缺失的行
df_anomaly = df_interp[sensor_cols].dropna()
# 标准化：(原值-均值)/标准差，消除各传感器量纲、幅值差异
df_anomaly_std = (df_anomaly - df_anomaly.mean()) / df_anomaly.std()

# 初始化孤立森林模型
# contamination=0.01：预设异常点占全部样本的1%
# random_state=42：固定随机种子，保证每次运行结果一致
model = IsolationForest(contamination=0.01, random_state=42)
# 用标准化后的多维数据训练模型
model.fit(df_anomaly_std)
# 模型预测：返回 1（正常）、-1（异常）
preds = model.predict(df_anomaly_std)

# 把预测结果存入新列anomaly
df_anomaly["anomaly"] = preds
# 映射转换：1→0（正常），-1→1（异常），方便后续筛选
df_anomaly["anomaly"] = df_anomaly["anomaly"].map({1: 0, -1: 1})


# 创建plotly画布
fig = go.Figure()

# 添加原始温度时序曲线
fig.add_trace(go.Scatter(
    x=df_anomaly.index,
    y=df_anomaly["TE_8303.AV_0"],
    mode='lines',
    name='Temperature'
))

# 添加异常点标记：只取出anomaly=1的点，画红色叉号
fig.add_trace(go.Scatter(
    x=df_anomaly[df_anomaly["anomaly"] == 1].index,
    y=df_anomaly[df_anomaly["anomaly"] == 1]["TE_8303.AV_0"],
    mode='markers',
    name='Anomalies',
    marker=dict(color='red', size=6, symbol='x')
))

# 设置图表布局
fig.update_layout(
    title="Anomaly Detection using Isolation Forest",
    xaxis_title="Time",
    yaxis_title="Temperature (TE_8303.AV_0)",
    height=500
)

# 展示交互式图
fig.show()
